In [ ]:
!nvidia-smi

In [ ]:
!pip install chromadb
!pip install unstructured

In [ ]:
!pip install langchain
!pip install InstructorEmbedding

In [ ]:
!pip install tiktoken
!pip install openai
!pip install transformers[sentencepiece]
!pip install sentencepiece

In [ ]:
!pip install -U sentence-transformers

In [ ]:
!pip install gradio

In [ ]:
import os
os.environ['HUGGINGFACEHUB_API_TOKEN']=''
os.environ['OPENAI_API_KEY'] =''

In [ ]:
from langchain.embeddings import HuggingFaceInstructEmbeddings
from langchain.document_loaders import DirectoryLoader
from langchain.embeddings import HuggingFaceHubEmbeddings
from langchain.vectorstores import Chroma
from langchain.text_splitter import CharacterTextSplitter
from langchain.chains.question_answering import load_qa_chain
from langchain.chains import VectorDBQA
from langchain.llms import OpenAI,HuggingFaceHub
from langchain.embeddings.openai import OpenAIEmbeddings
from InstructorEmbedding import INSTRUCTOR
from transformers import pipeline
import gradio as gr

In [ ]:
from langchain.llms import OpenAI
from langchain.chains import LLMChain, ConstitutionalChain
from langchain.chains.constitutional_ai.models import ConstitutionalPrinciple
from langchain import PromptTemplate

In [ ]:
persist_directory = ''
EMBEDDING_MODEL='text-embedding-ada-002'
embeddings=OpenAIEmbeddings(openai_api_key=os.environ['OPENAI_API_KEY'])
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embeddings)

In [ ]:
def chat(chat_history, user_input):
  global vectordb
  # docs=doc_store1.similarity_search(user_input)
  llm = OpenAI(model_name='text-davinci-003',openai_api_key=os.environ['OPENAI_API_KEY'],temperature=0)
  chain=load_qa_chain(llm,chain_type='stuff')
  # chain = load_qa_chain(llm, chain_type="stuff")
  # result=chain({"input_documents": docs, "question": '''Give relevant answer for the given question specific to Indian Legal Documents.If you don't know
  # the answer,Say I don't Know. \n'''+'Question:'+user_input},return_only_outputs=True)
  prompt='''Your task is to answer a question as a legal assistant to the best of your abilities, using the context given in the document. If the country is not mentioned in the question, your response should be related to India. You have knowledge of all laws and legal judgements of India. Be detailed in your answer, provide relevant sections and caselaws in your answer only if you are confident that they are correct.
  Note that if you do not know the answer, it is acceptable to say "Sorry, I don't know."
  {context}
  {Question:}
  '''
  context_docs=vectordb.similarity_search(user_input)
  result=chain.run(input_documents=context_docs,question=prompt+user_input)
  bot_response = result
  #print(bot_response)
  response = ""
  for letter in ''.join(bot_response): #[bot_response[i:i+1] for i in range(0, len(bot_response), 1)]:
      response += letter + ""
      yield chat_history + [(user_input, response)]

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown('# LEGAL BOT')
    with gr.Tab(""):
          chatbot = gr.Chatbot()
          message = gr.Textbox (placeholder="Ask me a question")
          message.submit(chat, [chatbot, message], chatbot)
demo.queue().launch(debug = True)


# OpenAI Model

In [ ]:
!pip install pinecone-client

In [ ]:
from langchain.document_loaders import DirectoryLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.chains.question_answering import load_qa_chain
from langchain.llms import OpenAI

In [ ]:
from langchain.vectorstores import Pinecone
from langchain.embeddings.openai import OpenAIEmbeddings
import pinecone

In [ ]:
EMBEDDING_MODEL='text-embedding-ada-002'
embeddings=OpenAIEmbeddings(openai_api_key=os.environ['OPENAI_API_KEY'])

In [ ]:
pinecone.init(api_key="9ed3ba66-f0d5-452a-96cf-dd28206e92ab", environment="northamerica-northeast1-gcp")
index_name='sample-index'

In [ ]:
index=pinecone.Index(index_name)
index.describe_index_stats()


In [ ]:
doc_store1=Pinecone.from_existing_index(index_name,embeddings)

In [ ]:
pip install gradio

In [ ]:
import gradio as gr

In [ ]:
def chat(chat_history, user_input):
  global doc_store1
  docs=doc_store1.similarity_search(user_input)
  llm=OpenAI(temperature=0,openai_api_key=os.environ['OPENAI_API_KEY'])
  chain = load_qa_chain(llm, chain_type="stuff")
  result=chain({"input_documents": docs, "question": '''Give relevant answer for the given question specific to Indian Legal Documents.If you don't know
  the answer,Say I don't Know. \n'''+'Question:'+user_input},return_only_outputs=True)
  bot_response = result['output_text']
  #print(bot_response)
  response = ""
  for letter in ''.join(bot_response): #[bot_response[i:i+1] for i in range(0, len(bot_response), 1)]:
      response += letter + ""
      yield chat_history + [(user_input, response)]

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown('# LEGAL BOT')
    with gr.Tab(""):
          chatbot = gr.Chatbot()
          message = gr.Textbox (placeholder="Ask me a question")
          message.submit(chat, [chatbot, message], chatbot)
demo.queue().launch(debug = True)


In [ ]:
!pip install flash-attn --no-build-isolation

## LLAMA Testing

In [ ]:
!pip install -qU transformers accelerate einops langchain xformers bitsandbytes faiss-gpu sentence_transformers

In [ ]:
!pip install replicate

In [ ]:
import os
os.environ['REPLICATE_API_TOKEN']='r8_GZTQL06XNxoZfs0ngLnwYXpOMDwWeA41FVnfE'
import numpy as np
import pandas as pd
import csv
import replicate

In [ ]:
test_data=pd.read_csv('/content/drive/MyDrive/Legal_GPT/Question-Answer-CSV/test_data_openai.csv')

In [ ]:
test_data.head()

In [ ]:
from tqdm import tqdm

In [ ]:
system_prompt='''
You are an honest legal advisor. Your task is to answer a question as a legal assistant to the best of your abilities based on the context provided. If the country is not mentioned in the question, your response should be related to India.
  You have knowledge of all laws and legal judgements of India. Be detailed in your answer, provide relevant sections and caselaws in your answer only if you are confident that they are correct.
  If you are unsure about an answer, truthfully say "I don't know"
'''

In [ ]:
rows=[]
for i in tqdm(range(len(test_data))):
  question=test_data['question'][i].strip()
  context=test_data['context'][i].strip()
  title=test_data['title'][i].strip()
  ground_truth=test_data['ground_truth'][i].strip()
  PROMPT=f"""
  Question:{question}
  """
  output = replicate.run(
      "meta/llama-2-70b-chat:02e509c789964a7ea8736978a43525956ef40397be9033abf9fd2badfe68c9e3",
      input={"prompt": PROMPT,
            "system_prompt": system_prompt,
            "max_new_tokens":1000,
            "temperature":0.01
            }

  )
  answer=""
  for item in output:
    answer+=item+''
  rows.append([title,question,ground_truth,answer,0])

In [ ]:
rows=np.array(rows)
result_df=pd.DataFrame(rows,columns=['title','question','ground_truth','answer_generated','score'])

In [ ]:
result_df.head()

In [ ]:
result_df.to_csv('/content/drive/MyDrive/Legal_GPT/Output/llama2+opensource_embedding.csv')